In [2]:
import os
import glob
import pdfplumber
import pandas as pd
import numpy as np

# 1. Setup absolute paths using your updated project directory path
base_project_dir = "/home/prasad/Ganga-River-stem_ES-Project"
pdf_folder = os.path.join(base_project_dir, "data/raw/cpcb")
pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))

target_stations = {
    "10148": "Ganga at Har-Ki-Pauri Ghat",
    "1067":  "Ganga at Jajmau Bridge",
    "1049":  "Ganga at Sangam",
    "1071":  "Ganga at Malviya Bridge",
    "2552":  "Ganga at Gandhi Ghat"
}

extracted_records = []
print(f"Starting 100% complete scan for {len(pdf_files)} CPCB reports...\n")

# 2. Comprehensive page-by-page extraction loop
for i, pdf_path in enumerate(pdf_files, 1):
    filename = os.path.basename(pdf_path)
    print(f"[{i}/{len(pdf_files)}] Scanning ALL pages of: {filename}...")
    
    # Extract the 4-digit year from the filename string structure safely
    year_digits = "".join(filter(str.isdigit, filename))
    year = year_digits if len(year_digits) == 4 else "Unknown"
    
    file_rows = 0
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                text = page.extract_text()
                if not text: 
                    continue
                    
                lines = text.split("\n")
                for line in lines:
                    for code, clean_name in target_stations.items():
                        if code in line:
                            extracted_records.append({
                                "station_code": code,
                                "station_name": clean_name,
                                "year": year,
                                "page_number": page_num,
                                "raw_line_text": line,
                                "file_source": filename
                            })
                            file_rows += 1
                            break
                            
        print(f"   -> Finished {filename}! Found {file_rows} records.\n")
    except Exception as e:
        print(f"   -> Error parsing {filename}: {e}\n")

# 3. Structural verification and clean compilation
df_extracted = pd.DataFrame(extracted_records)
if not df_extracted.empty:
    df_extracted = df_extracted.drop_duplicates(subset=["station_code", "year", "raw_line_text"])
    
print(f"\nScan Complete! Total verified rows captured: {len(df_extracted)}")

if not df_extracted.empty:
    # Save target directly inside your project structure workspace environment folder
    processed_folder = os.path.join(base_project_dir, "data/processed")
    os.makedirs(processed_folder, exist_ok=True)
    
    output_path = os.path.join(processed_folder, "wq_raw_extracted.csv")
    df_extracted.to_csv(output_path, index=False)
    print(f"Complete raw data matrix safely written to: {output_path}")
    
    print("\nBreakdown of records captured per year:")
    display(df_extracted.groupby("year").size().reset_index(name="rows_captured"))


Starting 100% complete scan for 13 CPCB reports...

[1/13] Scanning ALL pages of: NWMP_DATA_2015.pdf...
   -> Finished NWMP_DATA_2015.pdf! Found 6 records.

[2/13] Scanning ALL pages of: WQuality_River-Data-2018.pdf...
   -> Finished WQuality_River-Data-2018.pdf! Found 5 records.

[3/13] Scanning ALL pages of: WQuality_River-Data-2019.pdf...
   -> Finished WQuality_River-Data-2019.pdf! Found 6 records.

[4/13] Scanning ALL pages of: WQuality_River-Data-2020.pdf...
   -> Finished WQuality_River-Data-2020.pdf! Found 5 records.

[5/13] Scanning ALL pages of: WQuality_River-Data-2021.pdf...
   -> Finished WQuality_River-Data-2021.pdf! Found 5 records.

[6/13] Scanning ALL pages of: WQuality_River-Data-2022.pdf...
   -> Finished WQuality_River-Data-2022.pdf! Found 5 records.

[7/13] Scanning ALL pages of: Water_Quality_Rivers_2017.pdf...
   -> Finished Water_Quality_Rivers_2017.pdf! Found 5 records.

[8/13] Scanning ALL pages of: Water_Quality_data_of_River_2016.pdf...
   -> Finished Water_

,year,rows_captured
0,2012,2
1,2015,6
2,2016,4
3,2017,5
4,2018,5
5,2019,6
6,2020,5
7,2021,5
8,2022,5
9,2024,6


In [3]:
import pandas as pd
import os

# Load your extracted dataset
csv_path = "/home/prasad/Ganga-River-stem_ES-Project/data/processed/wq_raw_extracted.csv"
df = pd.read_csv(csv_path)

# Correct the 'Unknown' years using the filename metadata
def fix_unknown_years(row):
    if row['year'] == 'Unknown' or pd.isna(row['year']):
        fname = str(row['file_source'])
        if '2012' in fname: return 2012
        if '2013' in fname: return 2013
        if '2014' in fname: return 2014
    return row['year']

df['year'] = df.apply(fix_unknown_years, axis=1)

# Save back the updated clean baseline matrix
df.to_csv(csv_path, index=False)
print("Updated year references inside wq_raw_extracted.csv successfully!")

print("\n--- Fixed Year Summary ---")
print(df.groupby("year").size().reset_index(name="rows_captured"))

print("\n--- Sample Raw Line Strings (Need this for Step 2) ---")
for idx, text in enumerate(df['raw_line_text'].head(3), 1):
    print(f"Sample {idx}: {text}")


Updated year references inside wq_raw_extracted.csv successfully!

--- Fixed Year Summary ---
    year  rows_captured
0   2013              4
1   2014              4
2   2012              2
3   2015              6
4   2016              4
5   2017              5
6   2018              5
7   2019              6
8   2020              5
9   2021              5
10  2022              5
11  2024              6

--- Sample Raw Line Strings (Need this for Step 2) ---
Sample 1: 1067 GANGA AT KANPUR U/S (RANIGHAT), UTTAR 12 33 6.4 10 7.3 8.9 238 402 2.2 4 0 0.4 2200 4000 2800 6300
Sample 2: 1049 GANGA AT ALLAHABAD D/S UTTAR 20 30.6 7 8.7 7.4 8.4 365 499 3.4 4.8 0 1.5 17000 26000 32000 39000
Sample 3: 1071 GANGA AT VARANASI D/S (MALVIYA UTTAR 19 26 7 7.8 8.4 8.8 382 430 4.2 5.3 0 0 23000 33000 40000 49000


In [4]:
import pandas as pd
import re
import os

# 1. Load data
csv_path = "/home/prasad/Ganga-River-stem_ES-Project/data/processed/wq_raw_extracted.csv"
df = pd.read_csv(csv_path)

cleaned_rows = []

# 2. Extract numeric trailing sequences
for idx, row in df.iterrows():
    text = str(row['raw_line_text'])
    
    # Isolate all standard decimal/integer numbers from the tail end of the text line string
    numbers = re.findall(r'\b\d+\.?\d*\b', text)
    
    # The first item found by regex might be the station code itself, remove it if it matches
    if numbers and numbers[0] == str(row['station_code']):
        numbers = numbers[1:]
        
    # We require exactly 16 values (8 parameters x Min/Max pairs)
    if len(numbers) >= 16:
        # Grab the last 16 values explicitly to bypass any trailing names or states
        data_vals = numbers[-16:]
        
        cleaned_rows.append({
            "station_code": row['station_code'],
            "station_name": row['station_name'],
            "year": row['year'],
            "temp_min": float(data_vals[0]),  "temp_max": float(data_vals[1]),
            "do_min":   float(data_vals[2]),  "do_max":   float(data_vals[3]),
            "ph_min":   float(data_vals[4]),  "ph_max":   float(data_vals[5]),
            "cond_min": float(data_vals[6]),  "cond_max": float(data_vals[7]),
            "bod_min":  float(data_vals[8]),  "bod_max":  float(data_vals[9]),
            "nitrate_min": float(data_vals[10]), "nitrate_max": float(data_vals[11]),
            "fecal_col_min": float(data_vals[12]), "fecal_col_max": float(data_vals[13]),
            "total_col_min": float(data_vals[14]), "total_col_max": float(data_vals[15])
        })
    else:
        print(f"Skipping row index {idx} due to unexpected matrix shape structure layout: {text}")

# 3. Create a clean matrix dataset file
df_clean = pd.DataFrame(cleaned_rows)
output_clean_path = "/home/prasad/Ganga-River-stem_ES-Project/data/processed/wq_clean_final.csv"
df_clean.to_csv(output_clean_path, index=False)

print(f"\nSuccess! Clean dataset saved with {len(df_clean)} records.")
print(f"Location: {output_clean_path}")
display(df_clean.head(3))


Skipping row index 4 due to unexpected matrix shape structure layout: LAKSHDWEEP 29 31 7 7.5 1067 1325 4000 4000 1000 348000
Skipping row index 5 due to unexpected matrix shape structure layout: HERO CYCLE, 15 15 7 7.4 1067 1067 15 15
Skipping row index 6 due to unexpected matrix shape structure layout: 10148 UTTARAKHAND 16.0 21.0 8.8 10.0 7.4 8.4 _ _ 1.0 1.0 _ _ _ _ 60 170
Skipping row index 7 due to unexpected matrix shape structure layout: 1067 15.0 32.0 6.0 11.6 7.7 8.6 216 434 3.1 4.2 _ _ 2200 3400 4000 5800
Skipping row index 11 due to unexpected matrix shape structure layout: 10148 UTTARAKHAND 17 22 9.0 10.2 7.5 8.4 121 155 1.0 1.0 __ __ 26 50 40 110 - -
Skipping row index 12 due to unexpected matrix shape structure layout: 1067 16 32 5.8 10.3 7.8 8.7 203 368 2.5 4.0 __ __ 1700 3400 3900 5800 - -
Skipping row index 32 due to unexpected matrix shape structure layout: 10148 UTTARAKHAND 15 24 8.6 9.8 7 7.6 1 1 70 170
Skipping row index 36 due to unexpected matrix shape structure la

,station_code,station_name,year,temp_min,temp_max,do_min,do_max,ph_min,ph_max,cond_min,cond_max,bod_min,bod_max,nitrate_min,nitrate_max,fecal_col_min,fecal_col_max,total_col_min,total_col_max
0,1067,Ganga at Jajmau Bridge,2015,12.0,33.0,6.4,10.0,7.3,8.9,238.0,402.0,2.2,4.0,0.0,0.4,2200.0,4000.0,2800.0,6300.0
1,1049,Ganga at Sangam,2015,20.0,30.6,7.0,8.7,7.4,8.4,365.0,499.0,3.4,4.8,0.0,1.5,17000.0,26000.0,32000.0,39000.0
2,1071,Ganga at Malviya Bridge,2015,19.0,26.0,7.0,7.8,8.4,8.8,382.0,430.0,4.2,5.3,0.0,0.0,23000.0,33000.0,40000.0,49000.0
